In [28]:
gene_syns = '/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/data/gene/genes.syn'
genes_out = '/mnt/raidbio2/extproj/projekte/textmining/mirnaTextmining/mirClassification/data/synonyms/final/genes_filtered.syn'

In [29]:
from collections import defaultdict

In [30]:
id_to_syns = defaultdict(list)

In [31]:
with open(gene_syns, 'r') as f:
    for line in f:
        syn_id, syn_line = line.split(':', 1)
        syn_id = syn_id.strip()
        syn_line = syn_line.strip()
        syns = syn_line.split('|')
        id_to_syns[syn_id].extend(syn_line.split('|'))

In [ ]:
import re

MIRNA_LIKE = re.compile(
    r'^(?:[a-z]{2,4}[-_ ])?'                                # species prefix: hsa-, mmu-, dme-
    r'(?:'
    r'(?:micro[- ]?RNA|miRNA|miRN|miR|let|lin|lsy|iab)'     # miRNA token
    r'(?:[-_ ]?(?:mir|let|lin|lsy|iab))?'                   # second token: Mirlet7, mir-iab-4
    r'[-_ ]?\d'                                             # separator and first digit
    r'|'
    r'(?:(?:micro[- ]?RNA|miRNA|miR)[-_ ]?)?bantam\b'       # bantam has no number
    r'|'
    r'(?:micro[- ]?RNA|miRNA|miRN|miR|let|lin|lsy|iab|mir-iab)$'   # bare token as the whole name
    r')',
    re.IGNORECASE,
)

# Names that are only a number with an optional letter, e.g. "29b", "10A", "29beta", "2 9 beta".
# They come from fly cytological positions (Pp2A-29B) and collide with miRNA suffixes ("miR-29a and 29b").
NUMBER_LIKE = re.compile(r'^\d[\d ]*(?:[A-Za-z]|alpha|beta|gamma|delta)?$', re.IGNORECASE)

# Short word or Greek letter with a number: matches ordinary text and never identifies one gene.
#  - one or two letters, whitespace, number: "To 2", "or 10", "a 1". The fused form ("p53", "CD34",
#    "Ca2") is deliberately not covered.
#  - a spelled-out Greek letter, alone or with a number in any spacing: "alpha1", "beta-2", "alpha 2A"
#    (from "alpha1-adrenoceptor", "heregulin-beta1"; shared by 10-17 unrelated records).
GREEK = r'alpha|beta|gamma|delta|epsilon|kappa|sigma|zeta|theta|lambda|mu|omega'
SHORT_WORD_NUMBER = re.compile(
    r'^(?:'
    r'[A-Za-z]{1,2}\s\d+'
    r'|'
    rf'(?:{GREEK})(?:[ -]?\d+[a-z]?)?'
    r')$',
    re.IGNORECASE,
)

# Figure references in any spacing: yeast genes such as FIG2 and KAR5 (alias Fig3) match "Fig. 2" in the text.
# "Fig" alone and "Fig 2" are covered by the stoplist; this adds the fused and hyphenated forms ("FIG2", "Fig-1", "FIG2p").
FIGURE_REF = re.compile(r'^fig[ -]?\d+[a-z]?$', re.IGNORECASE)

In [33]:
from textmining.resources import GENE_STOPLIST
stopset = set()
with open(GENE_STOPLIST, 'r') as f:
    for line in f:
        stopset.add(line.strip().lower())

# <stoplist word> + number, e.g. "And 1" matches "and 1.5". Separators listed here are dropped:
# ' ' = "And 1", '-' = "And-1", '' = "And1"
STOPWORD_NUMBER_SEPS = {' '}
STOPWORD_NUMBER = re.compile(r'^([A-Za-z]+)([ -]?)\d+$')

def is_stopword_number(syn):
    m = STOPWORD_NUMBER.match(syn)
    return bool(m) and m.group(1).lower() in stopset and m.group(2) in STOPWORD_NUMBER_SEPS

In [34]:
def add_case_sensitivity(syn):
    if len(syn) < 8 and syn.isascii() and syn.isalpha():
        return syn + "@CASE_SENS"
    return syn

In [ ]:
filtered_id_to_syns = defaultdict(list)
for k, v in id_to_syns.items():
    kept = [add_case_sensitivity(s) for s in v if not (len(s) < 3 or MIRNA_LIKE.match(s) or NUMBER_LIKE.match(s) or SHORT_WORD_NUMBER.match(s) or s.lower() in stopset or is_stopword_number(s) or FIGURE_REF.match(s))]
    if len(kept) > 0:
        filtered_id_to_syns[k].extend(kept)

In [36]:
from textmining.synonym_utils import write_syn_file
write_syn_file(genes_out, 
               ((k, v) for k, v in filtered_id_to_syns.items()))